<a href="https://colab.research.google.com/github/physom-tech/csot-ml-astronomy/blob/main/week1_data_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import os
import random
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)



Using: cpu
Joined rows: 239100

Label counts:
label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64

Example rows:
   asset_id               objid gz2_class       label
0         3  587722981741363294        Sb      spiral
1         4  587722981741363323      Sc?l      spiral
2         5  587722981741559888        Er  elliptical
3         6  587722981741625481      Sc1t      spiral
4         7  587722981741625484        Sb      spiral
Linked images per class and split:
       elliptical  spiral  spiral_barred
train           0       0              0
val             0       0              0
test            0       0              0
train:    0 images  classes=[]
val  :    0 images  classes=[]
test :    0 images  classes=[]


In [2]:
from pathlib import Path # Added import for Path

RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2" / "images" # adjust if your JPGs landed one folder deeper
DATA_ROOT = Path("galaxy_data")        # we create train/val/test subfolders here
LABELS_URL = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"

# --- Download via Kaggle API (run once) ---
print("Please upload your kaggle.json file when prompted.")
from google.colab import files
files.upload()  # select kaggle.json

# Check if kaggle.json exists before moving it
if not Path('kaggle.json').exists():
    raise FileNotFoundError("kaggle.json not found. Please upload it.")

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle pandas

# Clean up existing RAW_ROOT to ensure a fresh start
!rm -rf {RAW_ROOT}
!mkdir -p {RAW_ROOT}

# Force re-download and unzip
!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images --force
!unzip -q -o galaxy-zoo-2-images.zip -d {RAW_ROOT}

# Removed the problematic line: !unzip -q -o {RAW_ROOT}/images_gz2.zip -d {IMAGES_DIR}

!wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS_URL}
!gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

# Verify contents of IMAGES_DIR
print(f"Contents of {IMAGES_DIR} after extraction:")
!ls -F {IMAGES_DIR}

Please upload your kaggle.json file when prompted.


FileNotFoundError: kaggle.json not found. Please upload it.

In [25]:
print(f"Contents of current directory (where galaxy-zoo-2-images.zip should be):")
!ls -F

Contents of current directory (where galaxy-zoo-2-images.zip should be):
galaxy_raw/  galaxy-zoo-2-images.zip  {RAW_ROOT}/  sample_data/


In [26]:
print(f"Contents of {RAW_ROOT} (after initial unzip):")
!ls -F {RAW_ROOT}

Contents of galaxy_raw (after initial unzip):
gz2_filename_mapping.csv  gz2_hart16.csv  images_gz2/


In [27]:
print(f"Listing contents of galaxy-zoo-2-images.zip to understand its structure:")
!unzip -l galaxy-zoo-2-images.zip | head -n 10

Listing contents of galaxy-zoo-2-images.zip to understand its structure:
Archive:  galaxy-zoo-2-images.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
 13166201  2021-01-26 16:34   gz2_filename_mapping.csv
    12897  2021-01-26 16:34   images_gz2/images/100.jpg
    11986  2021-01-26 16:34   images_gz2/images/1000.jpg
    18981  2021-01-26 16:34   images_gz2/images/10000.jpg
    15305  2021-01-26 16:34   images_gz2/images/100000.jpg
    14206  2021-01-26 16:34   images_gz2/images/100001.jpg
    12801  2021-01-26 16:34   images_gz2/images/100002.jpg


The output of the previous cell will help us understand if `images_gz2.zip` is contained within `galaxy-zoo-2-images.zip` or if the actual image files are directly inside a folder named `images_gz2`.

In [18]:
print("RAW_ROOT contents:", sorted(p.name for p in RAW_ROOT.iterdir()))
jpg_count = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print(f"Flat JPG count in {IMAGES_DIR}: {jpg_count:,}")

print("\nMapping CSV preview:")
print(pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3))

print("\nLabels CSV preview (note dr7objid — we rename to objid before merging):")
print(pd.read_csv(RAW_ROOT / "gz2_hart16.csv", nrows=3)[["dr7objid", "gz2_class"]])



RAW_ROOT contents: ['gz2_filename_mapping.csv', 'gz2_hart16.csv', 'images_gz2']
Flat JPG count in galaxy_raw/images_gz2: 0

Mapping CSV preview:
                objid    sample  asset_id
0  587722981736120347  original         1
1  587722981736579107  original         2
2  587722981741363294  original         3

Labels CSV preview (note dr7objid — we rename to objid before merging):
             dr7objid gz2_class
0  587732591714893851      Sc+t
1  588009368545984617      Sb+t
2  587732484359913515        Ei


In [17]:
def high_level_label(gz2_class: str):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, …) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None  # artifact / ambiguous
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    """Join Kaggle mapping (objid ↔ asset_id) with Hart et al. morphology labels."""
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    return df


def _link_image(src: Path, dst: Path) -> bool:
    """Symlink if possible; otherwise copy (some Drive setups block symlinks)."""
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src.resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(
    images_dir,
    df,
    out_root,
    per_class=200,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    """Create out_root/{train,val,test}//*.jpg for ImageFolder."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    images_dir = Path(images_dir)
    out_root = Path(out_root)
    summary = {}

    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)

        n = len(rows)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        n_test = n - n_train - n_val
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train : n_train + n_val],
            "test": rows.iloc[n_train + n_val :],
        }

        summary[label] = {}
        for split_name, split_rows in splits.items():
            linked = 0
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists() and _link_image(src, dst):
                    linked += 1
            summary[label][split_name] = linked
    return summary

df = load_labeled_table(
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
)
print("Joined rows:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nExample rows:")
print(df[["asset_id", "objid", "gz2_class", "label"]].head())



Joined rows: 239100

Label counts:
label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64

Example rows:
   asset_id               objid gz2_class       label
0         3  587722981741363294        Sb      spiral
1         4  587722981741363323      Sc?l      spiral
2         5  587722981741559888        Er  elliptical
3         6  587722981741625481      Sc1t      spiral
4         7  587722981741625484        Sb      spiral


In [12]:
PER_CLASS = 200  # increase once the pipeline works (e.g. 2000)

summary = build_split_imagefolder_layout(
    IMAGES_DIR,
    df,
    DATA_ROOT,
    per_class=PER_CLASS,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
)
print("Linked images per class and split:")
print(pd.DataFrame(summary).fillna(0).astype(int))

for split in ("train", "val", "test"):
    split_dir = DATA_ROOT / split
    classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir()) if split_dir.exists() else []
    n_imgs = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    print(f"{split:5s}: {n_imgs:4d} images  classes={classes}")


Linked images per class and split:
       elliptical  spiral  spiral_barred
train           0       0              0
val             0       0              0
test            0       0              0
train:    0 images  classes=[]
val  :    0 images  classes=[]
test :    0 images  classes=[]


In [20]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [28]:
train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"{name:5s}  n={len(ds):4d}  classes={ds.classes}")

print("class_to_idx:", train_ds.class_to_idx)


FileNotFoundError: [Errno 2] No such file or directory: 'galaxy_data/train'